# Differentiable Rendering: Fitting SDF Scenes to Images

jaxCAD's sphere-tracing renderer is fully differentiable — gradients flow through the
march loop, normal computation, and shading.  This notebook uses that to solve an
**inverse-rendering problem**: given a target photograph (`lada.png`), optimise a small
scene of SDF primitives so its rendering minimises pixel-space MSE against the photo.

The scene has a box body, a sphere roof, and two sphere wheels.  We optimise:
* primitive positions and radii
* per-primitive material colours (parameterised with sigmoid so they stay in [0, 1])
* background sky colour

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from PIL import Image

# Import the internal sphere-tracing primitives directly so we can build a
# fully-differentiable render function that stays in JAX (no numpy conversion).
from jaxcad.render.raymarch import _camera_rays, _render_pixel

In [ ]:
# ── Target image ────────────────────────────────────────────────────────────
# lada.png is RGBA; composite the car onto a neutral sky-blue background that
# matches the renderer background we will use during optimisation.
RES = (64, 96)  # (height, width) — aspect ≈ lada original (1192×1920)
SKY = (140, 152, 168)  # sky-blue background colour (uint8)

img_rgba = Image.open("lada.png").convert("RGBA")
bg = Image.new("RGB", img_rgba.size, SKY)
bg.paste(img_rgba.convert("RGB"), mask=img_rgba.split()[3])
target_pil = bg.resize((RES[1], RES[0]), Image.LANCZOS)  # PIL: (width, height)

target = jnp.array(np.array(target_pil) / 255.0, dtype=jnp.float32)  # (H, W, 3)

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(np.array(target))
ax.set_title(f"Target  ({RES[1]}×{RES[0]} px downsampled from 1920×1192)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Functional scene

For differentiable optimisation the scene must be a **pure function of the parameters** —
no Python-level branching on traced values, all JAX ops.  We implement the SDF
primitives inline and blend materials with a temperature-scaled softmax over distances.

In [ ]:
# ── Primitive SDFs (pure JAX) ─────────────────────────────────────────────
def sphere_sdf(p, center, radius):
    return jnp.linalg.norm(p - center) - radius


def box_sdf(p, center, half_size):
    q = jnp.abs(p - center) - half_size
    return jnp.linalg.norm(jnp.maximum(q, 0.0)) + jnp.minimum(jnp.max(q), 0.0)


def smooth_union(d1, d2, k=0.2):
    """Inigo Quilez smooth min, blends two surfaces over radius k."""
    h = jnp.clip(0.5 + 0.5 * (d2 - d1) / k, 0.0, 1.0)
    return d1 * (1.0 - h) + d2 * h - k * h * (1.0 - h)


# ── Scene builder ──────────────────────────────────────────────────────────
def build_scene(params):
    """Return (sdf, material_fn) closures parameterised by *params* dict.

    Scene layout (loosely car-shaped):
      - flat box  → car body
      - sphere    → rounded roof
      - 2 spheres → wheels
      - infinite plane at y = -1.2  → ground
    """
    pos_body = params["pos_body"]  # (3,)
    size_body = params["size_body"]  # (3,) half-extents
    pos_roof = params["pos_roof"]  # (3,)
    r_roof = params["r_roof"]  # scalar
    pos_wl = params["pos_wl"]  # (3,)  left wheel
    pos_wr = params["pos_wr"]  # (3,)  right wheel
    r_wheel = params["r_wheel"]  # scalar

    # Colours live in unconstrained space; sigmoid maps them to (0, 1)
    col_body = jax.nn.sigmoid(params["col_body"])
    col_roof = jax.nn.sigmoid(params["col_roof"])
    col_wheel = jax.nn.sigmoid(params["col_wheel"])
    col_ground = jax.nn.sigmoid(params["col_ground"])

    def sdf(p):
        d_body = box_sdf(p, pos_body, size_body)
        d_roof = sphere_sdf(p, pos_roof, r_roof)
        d_wl = sphere_sdf(p, pos_wl, r_wheel)
        d_wr = sphere_sdf(p, pos_wr, r_wheel)
        d_ground = p[1] + 1.2

        d_car = smooth_union(d_body, d_roof, k=0.15)
        d_car = smooth_union(d_car, d_wl, k=0.06)
        d_car = smooth_union(d_car, d_wr, k=0.06)
        return smooth_union(d_car, d_ground, k=0.05)

    def material_fn(p):
        d_body = box_sdf(p, pos_body, size_body)
        d_roof = sphere_sdf(p, pos_roof, r_roof)
        d_wl = sphere_sdf(p, pos_wl, r_wheel)
        d_wr = sphere_sdf(p, pos_wr, r_wheel)
        d_ground = p[1] + 1.2

        # Softmax over *negative* distances → points closest to a primitive
        # get the highest weight for that primitive's colour.
        logits = jnp.array([-d_body, -d_roof, -d_wl, -d_wr, -d_ground]) * 12.0
        w = jax.nn.softmax(logits)  # (5,)

        # Both wheels share the same colour parameter
        all_cols = jnp.stack([col_body, col_roof, col_wheel, col_wheel, col_ground])  # (5, 3)
        color = w @ all_cols  # (3,)

        return {
            "color": color,
            "roughness": jnp.array(0.45),
            "metallic": jnp.array(0.05),
            "opacity": jnp.array(1.0),
            "ior": jnp.array(1.0),
        }

    return sdf, material_fn

In [ ]:
# ── Differentiable render function ────────────────────────────────────────
#
# We call _render_pixel / _camera_rays directly (bypassing the numpy
# conversion in `raymarch`) so the output stays a JAX array and gradients
# can flow back through the whole pipeline.

CAMERA_POS = jnp.array([0.0, 0.6, 6.0])
LOOK_AT = jnp.array([0.0, 0.0, 0.0])
FOV = 0.55
SCENE_DIST = float(jnp.linalg.norm(CAMERA_POS - LOOK_AT))

_LIGHT_DIRS = jnp.array([[0.45, 0.85, 0.25], [-0.3, 0.5, -0.2]], dtype=jnp.float32)
_LIGHT_DIRS = _LIGHT_DIRS / jnp.linalg.norm(_LIGHT_DIRS, axis=1, keepdims=True)
_LIGHT_COLS = jnp.array([[0.95, 0.88, 0.75], [0.3, 0.38, 0.55]], dtype=jnp.float32)


def diff_render(params, resolution=RES):
    """Render the scene and return a (H, W, 3) JAX float32 array.

    The function is fully differentiable with respect to *params* via
    jax.grad; gradients flow through the sphere-tracing loop, surface
    normals (computed via jax.grad of the SDF), shading, and material blending.
    """
    sdf, material_fn = build_scene(params)

    h, w = resolution
    edge_width = 2.0 * FOV / min(h, w) * SCENE_DIST
    bg = jax.nn.sigmoid(params["bg_color"])  # optimisable sky colour

    rays = _camera_rays(CAMERA_POS, LOOK_AT, resolution, FOV)

    def render_pixel(ray_dir):
        return _render_pixel(
            sdf,
            material_fn,
            CAMERA_POS,
            ray_dir,
            _LIGHT_DIRS,
            _LIGHT_COLS,
            max_steps=32,
            max_dist=15.0,
            shadow_steps=12,
            shadow_hardness=6.0,
            ambient=0.18,
            edge_width=edge_width,
            background_color=bg,
            refract_steps=0,
        )

    image = jax.vmap(render_pixel)(rays).reshape(h, w, 3)
    return jnp.clip(image ** (1.0 / 2.2), 0.0, 1.0)  # gamma correct, stay in JAX

In [ ]:
# ── Initial parameters ────────────────────────────────────────────────────
# A rough car-like pose — box body, sphere roof, two round wheels.
# Colours are in logit space: sigmoid(0) = 0.5 (mid-grey).

params0 = {
    # Geometry
    "pos_body": jnp.array([0.0, -0.05, 0.0]),
    "size_body": jnp.array([1.85, 0.45, 0.72]),
    "pos_roof": jnp.array([-0.15, 0.72, 0.0]),
    "r_roof": jnp.array(0.6),
    "pos_wl": jnp.array([-1.15, -0.68, 0.5]),
    "pos_wr": jnp.array([1.15, -0.68, 0.5]),
    "r_wheel": jnp.array(0.40),
    # Colours (logit space)
    "col_body": jnp.zeros(3),  # sigmoid(0) ≈ 0.5 mid-grey
    "col_roof": jnp.zeros(3),
    "col_wheel": jnp.full((3,), -2.0),  # sigmoid(-2) ≈ 0.12 dark
    "col_ground": jnp.array([0.3, 0.15, -0.2]),  # earthy brown-grey
    "bg_color": jnp.array([0.1, 0.25, 0.55]),  # sky blue in logit space
}

print("Rendering initial scene (first run triggers XLA compilation)...")
img0 = diff_render(params0)
print(f"Output shape: {img0.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(np.array(target))
axes[0].set_title("Target (lada.png)")
axes[0].axis("off")
axes[1].imshow(np.array(img0))
axes[1].set_title("Initial render")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Optimisation

We minimise mean-squared pixel error between the render and the target with Adam.

`jax.value_and_grad` differentiates through the entire pipeline — sphere-tracing,
surface normals (via a second `jax.grad` inside `_render_pixel`), Blinn–Phong shading,
and the softmax material blend — in one `jax.jit`-compiled call.

In [ ]:
def loss_fn(params):
    rendered = diff_render(params)
    return jnp.mean((rendered - target) ** 2)


# Compile once — subsequent calls are fast
grad_fn = jax.jit(jax.value_and_grad(loss_fn))

N_STEPS = 200
LR = 0.04
optimizer = optax.adam(learning_rate=LR)
opt_state = optimizer.init(params0)

params = params0
losses = []
snapshots = {}  # rendered images at selected steps

print(f"Optimising for {N_STEPS} steps (lr={LR}, Adam)")
print("First step triggers JIT compilation — may take a moment...")

for step in range(N_STEPS):
    val, grads = grad_fn(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    losses.append(float(val))

    if step in (0, 24, 49, 99, 149, 199):
        snapshots[step] = np.array(diff_render(params))

    if step % 25 == 0 or step == N_STEPS - 1:
        print(f"  step {step:3d}/{N_STEPS-1}  |  loss = {val:.5f}")

In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(losses, color="steelblue", linewidth=1.5)
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE Loss")
ax.set_title("Pixel-space MSE over optimisation")
ax.set_yscale("log")
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
# ── Optimisation trajectory ───────────────────────────────────────────────
snap_steps = sorted(snapshots.keys())
n_cols = len(snap_steps) + 1
fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5))

axes[0].imshow(np.array(target))
axes[0].set_title("Target")
axes[0].axis("off")

for ax, step in zip(axes[1:], snap_steps):
    ax.imshow(snapshots[step])
    ax.set_title(f"step {step}")
    ax.axis("off")

plt.suptitle("Optimisation trajectory", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── High-resolution final render ─────────────────────────────────────────
HIRES = (240, 360)
img_final = np.array(diff_render(params, resolution=HIRES))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

target_hires = np.array(target_pil.resize((HIRES[1], HIRES[0]), Image.LANCZOS)) / 255.0

axes[0].imshow(target_hires)
axes[0].set_title("Target (lada.png)")
axes[0].axis("off")

axes[1].imshow(np.array(diff_render(params0, resolution=HIRES)))
axes[1].set_title("Before optimisation")
axes[1].axis("off")

axes[2].imshow(img_final)
axes[2].set_title(f"After {N_STEPS} steps")
axes[2].axis("off")

plt.suptitle("Inverse rendering: fitting SDF scene to a photograph", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Initial loss: {losses[0]:.5f}")
print(f"Final loss:   {losses[-1]:.5f}")
print(f"Improvement:  {losses[0]/losses[-1]:.1f}×")